# Extract Local Thickness Data from Fiji Results

This notebook processes Fiji Local Thickness results and extracts blob statistics into CSV files.

In [1]:
# Block 1: Parameters

# Input/Output directories
INPUT_DIR = r"/Users/pavel/Downloads/007_local_thickness_results"
OUTPUT_DIR = r"/Users/pavel/Downloads/008_local_thickness_csv"

# Binning parameters
NUM_BINS = 50
BIN_MIN = 0
BIN_MAX = 100

# File pattern to match
FILE_PATTERN = "*_LocThk.tif"

In [2]:
# Block 2: Import libraries

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Block 3: Create output directory if it doesn't exist

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
print(f"Output directory ready: {output_path}")

Output directory ready: /Users/pavel/Downloads/008_local_thickness_csv


In [4]:
# Block 4: Function to process a single image

def process_image(image_path, num_bins, bin_min, bin_max):
    """
    Process a single Local Thickness image and extract binned statistics.
    
    Parameters:
    -----------
    image_path : Path
        Path to the input TIF image
    num_bins : int
        Number of bins for histogram
    bin_min : float
        Minimum value for binning
    bin_max : float
        Maximum value for binning
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with bin statistics
    """
    # Load image
    img = np.array(Image.open(image_path))
    
    # Extract non-zero values (blob pixels)
    blob_values = img[img > 0]
    
    # Create bins
    bin_edges = np.linspace(bin_min, bin_max, num_bins + 1)
    
    # Calculate histogram
    counts, _ = np.histogram(blob_values, bins=bin_edges)
    
    # Calculate total number of blob pixels
    total_pixels = len(blob_values)
    
    # Calculate percentages
    percentages = (counts / total_pixels * 100) if total_pixels > 0 else np.zeros_like(counts)
    
    # Create DataFrame
    df = pd.DataFrame({
        'bin_min': bin_edges[:-1],
        'bin_max': bin_edges[1:],
        'pixel_count': counts,
        'percentage': percentages
    })
    
    return df

print("Processing function defined!")

Processing function defined!


In [5]:
# Block 5: Process all images

input_path = Path(INPUT_DIR)

# Find all matching TIF files
image_files = list(input_path.glob(FILE_PATTERN))

print(f"Found {len(image_files)} images to process\n")

if len(image_files) == 0:
    print(f"WARNING: No files matching '{FILE_PATTERN}' found in {INPUT_DIR}")
    print("Please check the input directory and file pattern.")
else:
    # Process each image
    for img_path in tqdm(image_files, desc="Processing images"):
        try:
            # Process image
            df = process_image(img_path, NUM_BINS, BIN_MIN, BIN_MAX)
            
            # Create output filename (same as input but with .csv extension)
            output_filename = img_path.stem + ".csv"
            output_filepath = output_path / output_filename
            
            # Save to CSV
            df.to_csv(output_filepath, index=False)
            
            print(f"✓ Processed: {img_path.name} -> {output_filename}")
            
        except Exception as e:
            print(f"✗ Error processing {img_path.name}: {str(e)}")
    
    print(f"\nProcessing complete! Results saved to: {output_path}")

Found 4 images to process



Processing images: 100%|██████████| 4/4 [00:00<00:00, 17.42it/s]

✓ Processed: E29-ROI-001_membrane_LocThk.tif -> E29-ROI-001_membrane_LocThk.csv
✓ Processed: E29-ROI-000_membrane_LocThk.tif -> E29-ROI-000_membrane_LocThk.csv
✓ Processed: A15_ROI000_membrane_LocThk.tif -> A15_ROI000_membrane_LocThk.csv
✓ Processed: A15_ROI001_membrane_LocThk.tif -> A15_ROI001_membrane_LocThk.csv

Processing complete! Results saved to: /Users/pavel/Downloads/008_local_thickness_csv


In [6]:
# Block 6: Quick verification - show example of first processed file

csv_files = list(output_path.glob("*.csv"))

if csv_files:
    example_csv = csv_files[0]
    print(f"Example output from: {example_csv.name}\n")
    df_example = pd.read_csv(example_csv)
    print(df_example.head(10))
    print(f"\nTotal rows: {len(df_example)}")
    print(f"Total percentage: {df_example['percentage'].sum():.2f}%")
else:
    print("No CSV files found in output directory.")

Example output from: A15_ROI000_membrane_LocThk.csv

   bin_min  bin_max  pixel_count  percentage
0      0.0      2.0            0    0.000000
1      2.0      4.0         3331    2.195709
2      4.0      6.0         2741    1.806796
3      6.0      8.0         7052    4.648495
4      8.0     10.0        15013    9.896180
5     10.0     12.0        37600   24.784944
6     12.0     14.0        23883   15.743054
7     14.0     16.0        23735   15.645496
8     16.0     18.0        17112   11.279786
9     18.0     20.0         6123    4.036123

Total rows: 50
Total percentage: 100.00%
